In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [9]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [10]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardtLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3841378688812256
epoch:  1, loss: 0.197881817817688
epoch:  2, loss: 0.043744269758462906
epoch:  3, loss: 0.03975950554013252
epoch:  4, loss: 0.03789401426911354
epoch:  5, loss: 0.03673310950398445
epoch:  6, loss: 0.030918309465050697
epoch:  7, loss: 0.025599386543035507
epoch:  8, loss: 0.0133307334035635
epoch:  9, loss: 0.012417524121701717
epoch:  10, loss: 0.011751829646527767
epoch:  11, loss: 0.011249739676713943
epoch:  12, loss: 0.0109279565513134
epoch:  13, loss: 0.010710098780691624
epoch:  14, loss: 0.010588334873318672
epoch:  15, loss: 0.010579576715826988
epoch:  16, loss: 0.010399393737316132
epoch:  17, loss: 0.010283908806741238
epoch:  18, loss: 0.01026264950633049
epoch:  19, loss: 0.010145502164959908
epoch:  20, loss: 0.010071657598018646
epoch:  21, loss: 0.010049345903098583
epoch:  22, loss: 0.009971032850444317
epoch:  23, loss: 0.009921547956764698
epoch:  24, loss: 0.009897290728986263
epoch:  25, loss: 0.009843003004789352
epoch:  2

In [11]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9658911834056266
Test metrics:  R2 = 0.9606413917090649


In [12]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardtLS(model=model,batch_size=100)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5411818623542786
epoch:  1, loss: 0.4482762813568115
epoch:  2, loss: 0.1661786288022995
epoch:  3, loss: 0.1059766560792923
epoch:  4, loss: 0.07810935378074646
epoch:  5, loss: 0.05770759657025337
epoch:  6, loss: 0.045422930270433426
epoch:  7, loss: 0.03912436589598656
epoch:  8, loss: 0.036132801324129105
epoch:  9, loss: 0.03338088467717171
epoch:  10, loss: 0.025813717395067215
epoch:  11, loss: 0.024101877585053444
epoch:  12, loss: 0.016969939693808556
epoch:  13, loss: 0.014752009883522987
epoch:  14, loss: 0.013952719047665596
epoch:  15, loss: 0.013398929499089718
epoch:  16, loss: 0.012937625870108604
epoch:  17, loss: 0.012544657103717327
epoch:  18, loss: 0.01220595370978117
epoch:  19, loss: 0.011913317255675793
epoch:  20, loss: 0.011661496944725513
epoch:  21, loss: 0.011444369331002235
epoch:  22, loss: 0.0112588657066226
epoch:  23, loss: 0.011100676842033863
epoch:  24, loss: 0.010967545211315155
epoch:  25, loss: 0.010858284309506416
epoch:  26,

In [13]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8091106213221866
Test metrics:  R2 = 0.7992989729647624
